This notebook is for data preprocessing check.


Dropped Time

Removed features with more than 50% missing values

Imputed remaining missing values with median

Removed constant features

Performed stratified 70/15/15 split

Standardized features using training statistics

VAE Feature Engineering (New Strategy):
Semi-supervised Training: The VAE is trained exclusively on the "Pass" (-1) class to learn the latent representation of normal manufacturing processes.

Reconstruction Error Extraction: Calculated the Mean Squared Error (MSE) between the original input and VAE reconstruction. High MSE serves as a strong Anomaly Score for the "Fail" (1) class.

Feature Fusion: The final dataset combines the 100 standardized features with the 1D Reconstruction Error feature, providing a stronger signal for the subsequent XGBoost/SVM classifiers.


VAE data saved to ../data/preprocessed_data_outlier_vae_enhanced.pkl

In [153]:
#import package
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath(".."))
from src.utils import RANDOM_SEED, TRAIN_RATIO, VAL_RATIO, TEST_RATIO

In [154]:
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [155]:
df_raw = pd.read_csv("../data/uci-secom.csv")
df = df_raw.copy() #use copy to not change the origianl file

In [156]:
print("Shape:", df.shape)

Shape: (1567, 592)


In [157]:
#delete pass/fail for not data leak
# Remove Time from the input features.
# We use only sensor-related features for the first baseline experiment.
y = df["Pass/Fail"].copy()
X = df.drop(columns=["Pass/Fail"]).copy()

print("Before dropping Time:", X.shape)

X = X.drop(columns=["Time"]).copy()

print("After dropping Time:", X.shape)

Before dropping Time: (1567, 591)
After dropping Time: (1567, 590)


In [158]:
#drop the data that have missing ratio bigger than 50%
missing_ratio = X.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.5].index
print("Number of columns to drop:", len(cols_to_drop))

Number of columns to drop: 28


In [159]:
X = X.drop(columns=cols_to_drop).copy()
print("Shape after dropping high-missing columns:", X.shape)

Shape after dropping high-missing columns: (1567, 562)


In [160]:
#check the missing value percentage for the data that remains
print("Total missing values in X:", X.isnull().sum().sum())
missing_ratio_after = X.isnull().mean().sort_values(ascending=False)
missing_ratio_after.head(10)

Total missing values in X: 11683


385    0.456286
247    0.456286
112    0.456286
519    0.456286
566    0.174218
567    0.174218
568    0.174218
569    0.174218
563    0.174218
562    0.174218
dtype: float64

In [161]:
#fill the data with median (for each column)
X = X.fillna(X.median())
print("Total missing values after median imputation:", X.isnull().sum().sum())

Total missing values after median imputation: 0


In [162]:
#find the col that have same numbers in it 
constant_cols = [col for col in X.columns if X[col].nunique() == 1]

print("Number of constant features:", len(constant_cols))
print("Constant features:", constant_cols[:10])

Number of constant features: 116
Constant features: ['5', '13', '42', '49', '52', '69', '97', '141', '149', '178']


In [163]:
X = X.drop(columns=constant_cols).copy()
print("Shape after dropping constant features:", X.shape)

Shape after dropping constant features:

 (1567, 446)


In [164]:
print(y.value_counts())
print(y.value_counts(normalize=True))

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64
Pass/Fail
-1    0.933631
 1    0.066369
Name: proportion, dtype: float64


In [165]:
corr_matrix = X.corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

print("Highly correlated features to drop:", len(to_drop_corr))

X = X.drop(columns=to_drop_corr)

print("Shape after correlation filtering:", X.shape)

Highly correlated features to drop: 174
Shape after correlation filtering: (1567, 272)


In [166]:
print (RANDOM_SEED)

559


In [167]:
# First split: train vs temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_SEED,
    stratify=y
)

# Second split: validation vs test from the temporary set
temp_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=temp_test_ratio,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1096, 272)
X_val shape: (235, 272)
X_test shape: (236, 272)


In [168]:
print("Train label distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nValidation label distribution:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nTest label distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train label distribution:
Pass/Fail
-1    1023
 1      73
Name: count, dtype: int64
Pass/Fail
-1    0.933394
 1    0.066606
Name: proportion, dtype: float64

Validation label distribution:
Pass/Fail
-1    220
 1     15
Name: count, dtype: int64
Pass/Fail
-1    0.93617
 1    0.06383
Name: proportion, dtype: float64

Test label distribution:
Pass/Fail
-1    220
 1     16
Name: count, dtype: int64
Pass/Fail
-1    0.932203
 1    0.067797
Name: proportion, dtype: float64


In [169]:
# Compute variance from training set only
train_var = X_train.var()

threshold = 0.01
low_variance_cols = train_var[train_var < threshold].index.tolist()

print("Number of low variance features:", len(low_variance_cols))
print("Some low variance features:", low_variance_cols[:10])

# Drop the same columns from all splits
X_train = X_train.drop(columns=low_variance_cols).copy()
X_val = X_val.drop(columns=low_variance_cols).copy()
X_test = X_test.drop(columns=low_variance_cols).copy()

print("X_train shape after low variance filtering:", X_train.shape)
print("X_val shape after low variance filtering:", X_val.shape)
print("X_test shape after low variance filtering:", X_test.shape)

Number of low variance features: 97
Some low variance features: ['7', '8', '9', '10', '11', '17', '20', '30', '53', '54']
X_train shape after low variance filtering: (1096, 175)
X_val shape after low variance filtering: (235, 175)
X_test shape after low variance filtering: (236, 175)


In [170]:
# Make sure y_train is a pandas Series
y_train_series = pd.Series(y_train, index=X_train.index)

# Compute absolute correlation between each feature and y
feature_scores = {}

for col in X_train.columns:
    corr_value = X_train[col].corr(y_train_series)
    
    # Handle possible NaN
    if pd.isna(corr_value):
        corr_value = 0.0
        
    feature_scores[col] = abs(corr_value)

# Convert to DataFrame and sort
score_df = pd.DataFrame({
    "feature": list(feature_scores.keys()),
    "score": list(feature_scores.values())
}).sort_values(by="score", ascending=False)

# Select top 100 features
top_k = 100
selected_cols = score_df.head(top_k)["feature"].tolist()

print("Top 10 selected features:")
print(score_df.head(10))
print("\nNumber of selected features:", len(selected_cols))

# Apply the same selected columns to train/val/test
X_train = X_train[selected_cols].copy()
X_val = X_val[selected_cols].copy()
X_test = X_test[selected_cols].copy()

print("Shapes after top-100 correlation selection:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Top 10 selected features:
    feature     score
155     510  0.170844
39       59  0.159619
43       63  0.126515
130     431  0.121391
133     434  0.116371
18       28  0.113416
129     430  0.111884
99      196  0.111673
100     197  0.111116
121     336  0.110520

Number of selected features: 100
Shapes after top-100 correlation selection:
X_train: (1096, 100)
X_val: (235, 100)
X_test: (236, 100)


In [171]:
# Compute mean and standard deviation from the training set only
train_mean = X_train.mean()
train_std = X_train.std()

# Avoid division by zero just in case
train_std = train_std.replace(0, 1)

# Standardize train, validation, and test sets
X_train_scaled = (X_train - train_mean) / train_std
X_val_scaled = (X_val - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (1096, 100)
X_val_scaled shape: (235, 100)
X_test_scaled shape: (236, 100)


In [172]:
print("Mean of first 5 scaled training features:")
print(X_train_scaled.iloc[:, :5].mean())

print("\nStd of first 5 scaled training features:")
print(X_train_scaled.iloc[:, :5].std())

Mean of first 5 scaled training features:
510   -1.166950e-16
59    -8.103818e-18
63    -5.915787e-17
431    1.296611e-17
434   -2.593222e-17
dtype: float64

Std of first 5 scaled training features:
510    1.0
59     1.0
63     1.0
431    1.0
434    1.0
dtype: float64


In [173]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import pickle

# --- 1. 定義 VAE 模型結構 ---
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super(VAE, self).__init__()
        # Encoder: 將 100 維壓縮到 latent 空間
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        
        # Decoder: 從 latent 空間還原回 100 維
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

# --- 2. 準備數據與訓練 VAE ---
# 只挑選訓練集中的 Pass (-1) 樣本進行訓練 
X_train_pass = X_train_scaled[y_train == -1].values
train_tensor = torch.FloatTensor(X_train_pass)

input_dim = X_train_scaled.shape[1] # 應該是 100 [cite: 26]
vae = VAE(input_dim=input_dim, latent_dim=16)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)
criterion = nn.MSELoss()

print("開始訓練 VAE (僅使用 Pass 樣本)...")
vae.train()
for epoch in range(100):
    optimizer.zero_grad()
    recon_x, mu, logvar = vae(train_tensor)
    
    # 計算 Reconstruction Loss (MSE) 與 KL 散度
    recon_loss = criterion(recon_x, train_tensor)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / train_tensor.size(0)
    loss = recon_loss + 0.01 * kl_loss
    
    loss.backward()
    optimizer.step()
    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}")

# --- 3. 提取重建誤差 (Anomaly Score) ---
def get_reconstruction_error(model, data_scaled):
    model.eval()
    with torch.no_grad():
        inputs = torch.FloatTensor(data_scaled.values)
        outputs, _, _ = model(inputs)
        # 對每個樣本計算 Mean Squared Error (MSE)
        errors = torch.mean((inputs - outputs)**2, dim=1).numpy()
    return errors.reshape(-1, 1)

train_error = get_reconstruction_error(vae, X_train_scaled)
val_error = get_reconstruction_error(vae, X_val_scaled)
test_error = get_reconstruction_error(vae, X_test_scaled)

# --- 4. 特徵整合與儲存 ---
# 將 100 維特徵與 1 維重建誤差合併
column_names = list(X_train_scaled.columns) + ["vae_reconstruction_error"]

vae_enhanced_data = {
    "X_train": pd.DataFrame(np.hstack([X_train_scaled.values, train_error]), columns=column_names),
    "X_val": pd.DataFrame(np.hstack([X_val_scaled.values, val_error]), columns=column_names),
    "X_test": pd.DataFrame(np.hstack([X_test_scaled.values, test_error]), columns=column_names),
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
    "note": "Semi-supervised VAE: 100 scaled features + 1D reconstruction error (Trained on Pass class only)"
}

output_path = "../data/preprocessed_data_outlier_vae_enhanced.pkl"
with open(output_path, "wb") as f:
    pickle.dump(vae_enhanced_data, f)

print(f"\n資料處理完成！已儲存至: {output_path}")
print(f"最終特徵維度: {vae_enhanced_data['X_train'].shape[1]}")

開始訓練 VAE (僅使用 Pass 樣本)...
Epoch [20/100], Loss: 0.9275
Epoch [40/100], Loss: 0.9258
Epoch [60/100], Loss: 0.9220
Epoch [80/100], Loss: 0.9014
Epoch [100/100], Loss: 0.8624

資料處理完成！已儲存至: ../data/preprocessed_data_outlier_vae_enhanced.pkl
最終特徵維度: 101


In [174]:
import pickle
import pandas as pd
import numpy as np

# 1. 將 100 維原始特徵與 1 維標準化後的誤差合併
# 請確認這裡使用的是 train_error_final (如果你跑的是優化版)
X_train_final = np.hstack([X_train_scaled.values, train_error_final])
X_val_final = np.hstack([X_val_scaled.values, val_error_final])
X_test_final = np.hstack([X_test_scaled.values, test_error_final])

# 2. 定義欄位名稱
column_names = list(X_train_scaled.columns) + ["vae_anomaly_score"]

# 3. 封裝並儲存
vae_enhanced_data = {
    "X_train": pd.DataFrame(X_train_final, columns=column_names),
    "X_val": pd.DataFrame(X_val_final, columns=column_names),
    "X_test": pd.DataFrame(X_test_final, columns=column_names),
    "y_train": y_train, # [cite: 4]
    "y_val": y_val,
    "y_test": y_test,
    "note": "Optimized VAE: Latent=4, Log-transformed & Scaled MSE (Trained on Pass only)"
}

# 儲存檔案
output_path = "../data/preprocessed_data_outlier_vae_enhanced.pkl"
with open(output_path, "wb") as f:
    pickle.dump(vae_enhanced_data, f)

print(f"VAE 增強版資料已儲存至: {output_path}")

VAE 增強版資料已儲存至: ../data/preprocessed_data_outlier_vae_enhanced.pkl
